# Advanced Preprocessing

## Objective

In this notebook, we will practice:

### Imbalanced Data Handling
- SMOTE
- Random Undersampling
- Class Weights

### Time Series Preprocessing
- Resampling
- Lag Features
- Rolling Window Features

> A random `practice_target` is used only to learn the imbalance-handling workflow.
> It does not represent a real medical outcome.

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [4]:
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


# Imbalanced Data Handling

A practice target will be created with an intentionally imbalanced class distribution so that SMOTE, Undersampling, and Class Weights can be demonstrated.

In [5]:
np.random.seed(42)

df_imbalance = df.copy()

df_imbalance["practice_target"] = np.random.choice(
    [0, 1],
    size=len(df_imbalance),
    p=[0.90, 0.10]
)

df_imbalance["practice_target"].value_counts()

practice_target
0    270
1     29
Name: count, dtype: int64

In [6]:
X = df_imbalance.drop(
    columns=["practice_target"]
)

y = df_imbalance["practice_target"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (299, 10)
y Shape: (299,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Distribution:")
print(y_train.value_counts())

print("\nTesting Distribution:")
print(y_test.value_counts())

Training Distribution:
practice_target
0    216
1     23
Name: count, dtype: int64

Testing Distribution:
practice_target
0    54
1     6
Name: count, dtype: int64


# SMOTE

SMOTE creates synthetic minority-class observations using patterns from existing minority samples.

It should be applied to training data only.

In [8]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
practice_target
0    216
1     23
Name: count, dtype: int64

After SMOTE:
practice_target
0    216
1    216
Name: count, dtype: int64


# Random Undersampling

Random Undersampling balances the dataset by removing some majority-class observations.

In [9]:
undersampler = RandomUnderSampler(
    random_state=42
)

X_train_under, y_train_under = undersampler.fit_resample(
    X_train,
    y_train
)

print("Before Undersampling:")
print(y_train.value_counts())

print("\nAfter Undersampling:")
print(y_train_under.value_counts())

Before Undersampling:
practice_target
0    216
1     23
Name: count, dtype: int64

After Undersampling:
practice_target
0    23
1    23
Name: count, dtype: int64


# Class Weights

Class Weights keep the dataset unchanged but make mistakes on underrepresented classes more important during model training.

In [10]:
weighted_model = LogisticRegression(
    class_weight="balanced",
    max_iter=5000
)

weighted_model.fit(
    X_train,
    y_train
)

print("Model Trained With Balanced Class Weights")

Model Trained With Balanced Class Weights


In [11]:
imbalance_comparison = pd.DataFrame({
    "Method": [
        "Original Training",
        "SMOTE",
        "Random Undersampling"
    ],
    "Rows": [
        len(X_train),
        len(X_train_smote),
        len(X_train_under)
    ],
    "Class 0": [
        (y_train == 0).sum(),
        (y_train_smote == 0).sum(),
        (y_train_under == 0).sum()
    ],
    "Class 1": [
        (y_train == 1).sum(),
        (y_train_smote == 1).sum(),
        (y_train_under == 1).sum()
    ]
})

imbalance_comparison

,Method,Rows,Class 0,Class 1
0,Original Training,239,216,23
1,SMOTE,432,216,216
2,Random Undersampling,46,23,23


# Time Series Preprocessing

The Heart Failure dataset is not a Time Series dataset.

A small practice dataset will therefore be created to demonstrate:

- Resampling
- Lag Features
- Rolling Window Features

In [12]:
date_range = pd.date_range(
    start="2026-01-01",
    periods=14,
    freq="D"
)

time_data = pd.DataFrame({
    "date": date_range,
    "sales": [
        100, 120, 110, 140, 160, 150, 170,
        180, 175, 190, 200, 210, 205, 220
    ]
})

time_data

,date,sales
0,2026-01-01,100
1,2026-01-02,120
2,2026-01-03,110
3,2026-01-04,140
4,2026-01-05,160
5,2026-01-06,150
6,2026-01-07,170
7,2026-01-08,180
8,2026-01-09,175
9,2026-01-10,190


# Resampling

Resampling changes the time frequency of the data.

Here, daily sales will be converted into weekly totals.

In [13]:
time_indexed = time_data.set_index(
    "date"
)

time_indexed.head()

,sales
date,
2026-01-01,100
2026-01-02,120
2026-01-03,110
2026-01-04,140
2026-01-05,160


In [14]:
weekly_sales = (
    time_indexed["sales"]
    .resample("W")
    .sum()
)

weekly_sales

date
2026-01-04     470
2026-01-11    1225
2026-01-18     635
Freq: W-SUN, Name: sales, dtype: int64

# Lag Features

Lag Features use previous observations as new features.

A 1-day lag will store the previous day's sales.

In [15]:
time_features = time_data.copy()

time_features["sales_lag_1"] = (
    time_features["sales"]
    .shift(1)
)

time_features.head()

,date,sales,sales_lag_1
0,2026-01-01,100,NaN
1,2026-01-02,120,100.0
2,2026-01-03,110,120.0
3,2026-01-04,140,110.0
4,2026-01-05,160,140.0


In [16]:
time_features["sales_lag_2"] = (
    time_features["sales"]
    .shift(2)
)

time_features.head()

,date,sales,sales_lag_1,sales_lag_2
0,2026-01-01,100,NaN,NaN
1,2026-01-02,120,100.0,NaN
2,2026-01-03,110,120.0,100.0
3,2026-01-04,140,110.0,120.0
4,2026-01-05,160,140.0,110.0


# Rolling Window Features

Rolling Window Features summarize a moving group of recent observations.

A 3-day rolling mean will be created.

In [17]:
time_features["rolling_mean_3"] = (
    time_features["sales"]
    .rolling(window=3)
    .mean()
)

time_features.head(7)

,date,sales,sales_lag_1,sales_lag_2,rolling_mean_3
0,2026-01-01,100,NaN,NaN,NaN
1,2026-01-02,120,100.0,NaN,NaN
2,2026-01-03,110,120.0,100.0,110.000000
3,2026-01-04,140,110.0,120.0,123.333333
4,2026-01-05,160,140.0,110.0,136.666667
5,2026-01-06,150,160.0,140.0,150.000000
6,2026-01-07,170,150.0,160.0,160.000000


In [18]:
time_features["rolling_sum_3"] = (
    time_features["sales"]
    .rolling(window=3)
    .sum()
)

time_features.head(7)

,date,sales,sales_lag_1,sales_lag_2,rolling_mean_3,rolling_sum_3
0,2026-01-01,100,NaN,NaN,NaN,NaN
1,2026-01-02,120,100.0,NaN,NaN,NaN
2,2026-01-03,110,120.0,100.0,110.000000,330.0
3,2026-01-04,140,110.0,120.0,123.333333,370.0
4,2026-01-05,160,140.0,110.0,136.666667,410.0
5,2026-01-06,150,160.0,140.0,150.000000,450.0
6,2026-01-07,170,150.0,160.0,160.000000,480.0


In [19]:
time_features["rolling_std_3"] = (
    time_features["sales"]
    .rolling(window=3)
    .std()
)

time_features.head(7)


,date,sales,sales_lag_1,sales_lag_2,rolling_mean_3,rolling_sum_3,rolling_std_3
0,2026-01-01,100,NaN,NaN,NaN,NaN,NaN
1,2026-01-02,120,100.0,NaN,NaN,NaN,NaN
2,2026-01-03,110,120.0,100.0,110.000000,330.0,10.000000
3,2026-01-04,140,110.0,120.0,123.333333,370.0,15.275252
4,2026-01-05,160,140.0,110.0,136.666667,410.0,25.166115
5,2026-01-06,150,160.0,140.0,150.000000,450.0,10.000000
6,2026-01-07,170,150.0,160.0,160.000000,480.0,10.000000


In [20]:
time_features.isnull().sum()

date              0
sales             0
sales_lag_1       1
sales_lag_2       2
rolling_mean_3    2
rolling_sum_3     2
rolling_std_3     2
dtype: int64

In [21]:
time_features_clean = (
    time_features
    .dropna()
    .reset_index(drop=True)
)

time_features_clean.head()

,date,sales,sales_lag_1,sales_lag_2,rolling_mean_3,rolling_sum_3,rolling_std_3
0,2026-01-03,110,120.0,100.0,110.000000,330.0,10.000000
1,2026-01-04,140,110.0,120.0,123.333333,370.0,15.275252
2,2026-01-05,160,140.0,110.0,136.666667,410.0,25.166115
3,2026-01-06,150,160.0,140.0,150.000000,450.0,10.000000
4,2026-01-07,170,150.0,160.0,160.000000,480.0,10.000000


# Time-Based Split

Time Series data should usually preserve chronological order.

Past observations will be used for training and later observations for testing.

In [22]:
split_index = int(
    len(time_features_clean) * 0.80
)

train_time = time_features_clean.iloc[
    :split_index
]

test_time = time_features_clean.iloc[
    split_index:
]

print("Training Data:")
print(train_time)

print("\nTesting Data:")
print(test_time)

Training Data:
        date  sales  sales_lag_1  sales_lag_2  rolling_mean_3  rolling_sum_3  \
0 2026-01-03    110        120.0        100.0      110.000000          330.0   
1 2026-01-04    140        110.0        120.0      123.333333          370.0   
2 2026-01-05    160        140.0        110.0      136.666667          410.0   
3 2026-01-06    150        160.0        140.0      150.000000          450.0   
4 2026-01-07    170        150.0        160.0      160.000000          480.0   
5 2026-01-08    180        170.0        150.0      166.666667          500.0   
6 2026-01-09    175        180.0        170.0      175.000000          525.0   
7 2026-01-10    190        175.0        180.0      181.666667          545.0   
8 2026-01-11    200        190.0        175.0      188.333333          565.0   

   rolling_std_3  
0      10.000000  
1      15.275252  
2      25.166115  
3      10.000000  
4      10.000000  
5      15.275252  
6       5.000000  
7       7.637626  
8      12.583

In [23]:
summary = pd.DataFrame({
    "Technique": [
        "SMOTE",
        "Random Undersampling",
        "Class Weights",
        "Resampling",
        "Lag Features",
        "Rolling Window"
    ],
    "Purpose": [
        "Increase minority class",
        "Reduce majority class",
        "Change class importance",
        "Change time frequency",
        "Use past observations",
        "Summarize recent observations"
    ]
})

summary

,Technique,Purpose
0,SMOTE,Increase minority class
1,Random Undersampling,Reduce majority class
2,Class Weights,Change class importance
3,Resampling,Change time frequency
4,Lag Features,Use past observations
5,Rolling Window,Summarize recent observations


In [24]:
print("Original Dataset Shape:", df.shape)

print(
    "practice_target in Original Dataset:",
    "practice_target" in df.columns
)

Original Dataset Shape: (299, 10)
practice_target in Original Dataset: False


# Summary

In this notebook, we practiced:

## Imbalanced Data Handling

- SMOTE
- Random Undersampling
- Class Weights

## Time Series Preprocessing

- Resampling
- Lag Features
- Rolling Window Features
- Time-Based Train/Test Split

## Key Learnings

- SMOTE creates synthetic minority-class samples.
- Random Undersampling removes some majority-class samples.
- Class Weights keep the original rows but change class importance.
- SMOTE and Undersampling should be applied only to training data.
- Resampling changes time frequency.
- Lag Features use past observations.
- Rolling Window Features summarize recent observations.
- Lag and Rolling Features naturally create missing values at the beginning.
- Time Series data should generally preserve chronological order.
- Future information must not leak into past training data.

> The `practice_target` used for imbalance handling is randomly generated and does not represent a real medical outcome.